In [1]:
import numpy as np
import re

from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import CXGate,  PauliEvolutionGate, QAOAAnsatz, RZGate
from qiskit.quantum_info import SparsePauliOp


from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import InverseCancellation, CommutativeCancellation
from qopt_best_practices.transpilation.swap_cancellation_pass import SwapToFinalMapping

# from qiskit_aer import AerSimulator
# from qiskit_aer.backends.backendconfiguration import AerBackendConfiguration

from qiskit_qaoa.utils.transpiler_passes import ExtendedSwapStrategy, FindCommutingPauliEvolutionsMulti
from qiskit_qaoa.utils.commuting_gate_router_precompute_full_chain import CommutingGateRouterPrecompute

from qiskit_qaoa.hubo.graph_to_hubo_hamiltonian import graph_to_hubo_hamiltonian
from qiskit_qaoa.utils.gfa_utils import gfa_file_to_graph


In [2]:
def print_circuit_info(qc: QuantumCircuit, circuit_name):
    print(f'{circuit_name} has {qc.num_qubits} qubits')
    print(f'{circuit_name} has {qc.num_nonlocal_gates()} non-local gates and {qc.depth(lambda instr: len(instr.qubits) > 1)} non-local depth')
    print(f'{circuit_name} contains {list(qc.count_ops().keys())} gates.')

In [3]:
filepath = '/nfs/users/nfs_j/jc59/quantumwork/pangenome/data/test_N3_W4.gfa'
graph, n, V, T = gfa_file_to_graph(filepath, [2,1,1])
hamiltonian = graph_to_hubo_hamiltonian(graph, n, T, lamda=10, constraint_terms=1.0)
ess = ExtendedSwapStrategy.from_grid(n, T)
num_physical_qubits = ess._num_vertices
max_layers = 23

Keeping constraints at times: [0 1 2]


In [4]:
pm = PassManager(
    [
        FindCommutingPauliEvolutionsMulti(), 
        CommutingGateRouterPrecompute(
            ess,
            max_layers=max_layers,
        ),
        SwapToFinalMapping(),
        InverseCancellation(gates_to_cancel=[CXGate()]),
        CommutativeCancellation(basis_gates=["cx", "swap", "rz", "rzz"]),
        InverseCancellation(gates_to_cancel=[CXGate()]),
    ]
)

In [5]:
qc = QuantumCircuit(num_physical_qubits)
qc.append(PauliEvolutionGate(hamiltonian), range(num_physical_qubits))
tqc = pm.run(qc)

Max layers needed to apply swap decompose: 22
Applied interaction: (6, 10)
Applied interaction: (7, 11)
Applied interaction: (6, 7, 9, 10, 11)
Applied interaction: (6, 7, 8, 9, 10, 11)
final_interaction: (6, 7, 8, 9, 10, 11), applied_cx: [(6, 10), (7, 11), (11, 10), (10, 9), (8, 9)]
Applied interaction: (6, 7)
final_interaction: (6, 7), applied_cx: [(6, 7)]
Applied interaction: (7, 10, 11)
final_interaction: (7, 10, 11), applied_cx: [(6, 10)]
final_interaction: (), applied_cx: []
Applied interaction: (0, 1)
Applied interaction: (0, 1, 2, 3)
Applied interaction: (0, 1, 2, 3, 5)
Applied interaction: (0, 1, 2, 3, 4, 5)
final_interaction: (0, 1, 2, 3, 4, 5), applied_cx: [(0, 1), (3, 2), (2, 1), (1, 5), (4, 5)]
Applied interaction: (0, 4)
final_interaction: (0, 4), applied_cx: [(0, 4)]
Applied interaction: (1, 2, 3)
Applied interaction: (0, 1, 2, 3, 4)
final_interaction: (0, 1, 2, 3, 4), applied_cx: [(1, 0), (0, 4)]
final_interaction: (), applied_cx: []
Applied interaction: (3, 7)
Applied i

In [6]:
tqc.draw(fold=-1)

global phase: 2.1952
           ┌───────────┐                                                                                               ┌───┐     ┌──────────┐                           ┌───┐                                                                                                                                                                                                                                                                  ┌───┐┌────────────┐                                                               ┌───┐                                                                                                                                                                                                          ┌───┐                       ┌───┐                                                                                                                                                                                                                                                                                            ┌───┐                                                                              ┌───┐                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 ┌───┐                              ┌───┐     ┌───┐                                 ┌───┐              ┌───┐                                     ┌───┐                                                         
  q_0 -> 0 ┤ Rz(3.625) ├───■────────────────────────────────────────────────────────────────────────────────■──────────┤ X ├─────┤ Rz(1.25) ├──■─────────────────────■──┤ X ├──■────────■──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤ X ├┤ Rz(-0.625) ├──■─────────────────────────────────────────────────────■──────┤ X ├─────────────────────────────────────────────────────────────────■─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────■──┤ X ├──■─────────────────■──┤ X ├──────X──────────────────────────■─────────────────────■────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤ X ├──────■──────────────────────────────────────────────────────■────────────────┤ X ├─

In [7]:
print_circuit_info(tqc, 'TQC')

TQC has 12 qubits
TQC has 495 non-local gates and 261 non-local depth
TQC contains ['cx', 'rz', 'swap'] gates.
